# Practice 5: Memory and threads (and how to check a restart)

**The idea:** attach a *checkpointer* and LangGraph saves the state after every step. A `thread_id` in the config picks which saved conversation to continue. `InMemorySaver` keeps it inside the running process; `SqliteSaver` keeps it in a file.

**The exercise's "done when":** threads are isolated, and a chat started in one run continues in a *later run* of the script.

**What changed from the first version**
1. **The restart was never tested.** The SQLite cell ran in the same process as everything else, so it looked identical to `InMemorySaver`. A restart means a *new process*: a new kernel, or a new `python script.py`. The only way a notebook can start a new process is a `!python ...` call or a kernel restart. Steps 4 and 5 below do that for real.
2. **Re-running the SQLite cell changes the answer.** The file keeps what the last run saved, so re-running with the same thread id continues the old chat. I reproduced it: the same input run in a second process gave 4 messages, not 2. That is durability working, but the notebook as written hid it, and a notebook whose result depends on how many times you ran it is hard to trust.
3. **The bot did not read the user's message.** The reply was `this is msg count 0`, so you could not tell it saw anything. It now echoes the newest message, and `msg_count` became `turn_count` (it counts bot turns, and starts at 1 for the first reply).
4. **The history was printed but not read.** Raw config dicts are hard to match to events. Step 3 prints step number, `next` and message count, so each line explains what happened.
5. **Tests were missing.** The restart behaviour is now tested with real subprocesses.
6. **Smaller:** duplicate imports, `List` without a type (`list[str]`), the database file left in the working folder, and the connection never named. Put `*.db` in `.gitignore`.

## Step 1. The module and the one-run script

In [1]:
%%writefile echo_memory.py
import sqlite3
from dataclasses import dataclass, field
from typing import Annotated
import operator

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import StateGraph, START, END


@dataclass
class State:
    messages: Annotated[list[str], operator.add] = field(default_factory=list)
    turn_count: int = 0          # no reducer: each node run overwrites it


def echo_bot(state: State) -> dict:
    """Reads the newest message, echoes it back, and counts the turn."""
    turn = state.turn_count + 1
    return {
        "messages": [f"echo (turn {turn}): {state.messages[-1]}"],
        "turn_count": turn,
    }


def make_saver(kind: str = "memory", db_path: str = "chat_memory.db"):
    """'memory' forgets when the process ends. 'sqlite' keeps everything in a file."""
    if kind == "memory":
        return InMemorySaver()
    if kind == "sqlite":
        return SqliteSaver(sqlite3.connect(db_path, check_same_thread=False))
    raise ValueError(f"unknown saver kind: {kind!r}")


def build_graph(checkpointer=None):
    builder = StateGraph(State)
    builder.add_node("echo_bot", echo_bot)
    builder.add_edge(START, "echo_bot")
    builder.add_edge("echo_bot", END)
    return builder.compile(checkpointer=checkpointer)

Overwriting echo_memory.py


In [2]:
%%writefile chat_cli.py
"""One run of this script = one process. Run it twice to test a 'restart'.

    python chat_cli.py --saver sqlite --db chat.db --thread alice "hello"
    python chat_cli.py --saver sqlite --db chat.db --thread alice "and again"
    python chat_cli.py --saver sqlite --db chat.db --thread alice --show     # read only
"""
import argparse

from echo_memory import build_graph, make_saver

parser = argparse.ArgumentParser()
parser.add_argument("message", nargs="?", default=None)
parser.add_argument("--saver", choices=["memory", "sqlite"], default="memory")
parser.add_argument("--db", default="chat_memory.db")
parser.add_argument("--thread", default="demo")
parser.add_argument("--show", action="store_true", help="only print the saved state, send nothing")
args = parser.parse_args()

graph = build_graph(make_saver(args.saver, args.db))
config = {"configurable": {"thread_id": args.thread}}

if args.show:
    values = graph.get_state(config).values
    print("saved messages:", values.get("messages", []))
    print("saved turn_count:", values.get("turn_count", 0))
else:
    result = graph.invoke({"messages": [args.message]}, config)
    print("messages:", result["messages"])
    print("turn_count:", result["turn_count"])

Overwriting chat_cli.py


## Step 2. Two threads, in memory
Same graph, two thread ids. Thread `a` gets two messages, `b` gets one.

In [3]:
from echo_memory import build_graph, make_saver

graph = build_graph(make_saver("memory"))
a = {"configurable": {"thread_id": "a"}}
b = {"configurable": {"thread_id": "b"}}

graph.invoke({"messages": ["msg 1 for a"]}, a)
graph.invoke({"messages": ["msg 1 for b"]}, b)
print("a:", graph.invoke({"messages": ["msg 2 for a"]}, a)["messages"])
print("b:", graph.get_state(b).values["messages"])
print()
print("turn_count a:", graph.get_state(a).values["turn_count"], "| b:", graph.get_state(b).values["turn_count"])

a: ['msg 1 for a', 'echo (turn 1): msg 1 for a', 'msg 2 for a', 'echo (turn 2): msg 2 for a']
b: ['msg 1 for b', 'echo (turn 1): msg 1 for b']

turn_count a: 2 | b: 1


## Step 3. Read the history

Each `invoke` writes more than one checkpoint: one when your input arrives, one after each node runs. Reading `step` and `next` tells you which is which. The newest checkpoint is first.

- `next` is **the node that would run from this checkpoint**. Empty means the run is finished.
- First `invoke`: `step -1` is the empty start, `step 0` is "input saved" (about to run `echo_bot`), `step 1` is "after `echo_bot`", and the run is finished.
- Second `invoke` on the same thread: `step 2` is the new turn starting from the finished state, `step 3` is "input saved", `step 4` is "after `echo_bot`".
- Message counts go 0, 1, 2, then 2, 3, 4 because the reducer *adds* each input to what the thread already holds.

In [4]:
print(f"{'step':>4}  {'next':<14} {'messages':>8}  turn_count")
for s in graph.get_state_history(a):
    v = s.values
    print(f"{s.metadata['step']:>4}  {str(s.next):<14} {len(v.get('messages', [])):>8}  {v.get('turn_count', '-')}")

step  next           messages  turn_count
   4  ()                    4  2
   3  ('echo_bot',)         3  1
   2  ('__start__',)        2  1
   1  ()                    2  1
   0  ('echo_bot',)         1  -
  -1  ('__start__',)        0  -


## Step 4. The restart check, for real

Every `!python ...` line below is a **separate process**, exactly like closing a terminal and opening a new one. First with the memory saver: the second process knows nothing about the first.

In [5]:
import sys, tempfile, os
PY = sys.executable
workdir = tempfile.mkdtemp()                      # a throwaway folder for the database
db = os.path.join(workdir, "chat.db")

print("--- memory saver ---")
!{PY} chat_cli.py --saver memory --thread t "first run"
!{PY} chat_cli.py --saver memory --thread t "second run"

--- memory saver ---
messages: ['first run', 'echo (turn 1): first run']
turn_count: 1
messages: ['second run', 'echo (turn 1): second run']
turn_count: 1


Now the same two runs with the SQLite saver. The second process finds the first process's messages in the file.

In [6]:
print("--- sqlite saver ---")
!{PY} chat_cli.py --saver sqlite --db {db} --thread t "first run"
!{PY} chat_cli.py --saver sqlite --db {db} --thread t "second run"
print()
print("--- a third process only READS the saved state ---")
!{PY} chat_cli.py --saver sqlite --db {db} --thread t --show
print()
print("--- a different thread in the same file starts fresh ---")
!{PY} chat_cli.py --saver sqlite --db {db} --thread other "hello"

--- sqlite saver ---
messages: ['first run', 'echo (turn 1): first run']
turn_count: 1
messages: ['first run', 'echo (turn 1): first run', 'second run', 'echo (turn 2): second run']
turn_count: 2

--- a third process only READS the saved state ---
saved messages: ['first run', 'echo (turn 1): first run', 'second run', 'echo (turn 2): second run']
saved turn_count: 2

--- a different thread in the same file starts fresh ---
messages: ['hello', 'echo (turn 1): hello']
turn_count: 1


## Step 5. How to check a restart yourself

**A. Two terminals (the clearest way).** In a terminal, from the folder with these files:

```
python chat_cli.py --saver sqlite --db chat.db --thread alice "hello"
python chat_cli.py --saver sqlite --db chat.db --thread alice "and again"
python chat_cli.py --saver sqlite --db chat.db --thread alice --show
```

The second command prints 4 messages, and `--show` prints them without sending anything. Delete `chat.db` to start over. Repeat with `--saver memory`: every run starts empty.

**B. Restart the notebook kernel.** Run the cell below, restart the kernel, and run only the cells *after* the imports. The saved chat is still in the file. This works because the file is on disk, not because of anything in the kernel.

**C. Look inside the file** (next cell). You should see two tables, `checkpoints` and `writes`, and one row per saved checkpoint per thread.

**D. What proves it:** the second run printed `turn_count: 2` and included the first run's messages, and a third process could read them. If it printed `turn_count: 1`, the state was lost.

**The common ways a "restart" test lies to you:**
- Everything runs in one process (the original notebook).
- The path is `":memory:"` or a different relative path each time, so each run opens a different empty database.
- You reused an old database file and did not notice the chat was already there.

In [7]:
import sqlite3
con = sqlite3.connect(db)
print("tables     :", [r[0] for r in con.execute("select name from sqlite_master where type='table'")])
print("checkpoints:", con.execute("select thread_id, count(*) from checkpoints group by thread_id").fetchall())
con.close()

tables     : ['checkpoints', 'writes']
checkpoints: [('other', 3), ('t', 6)]


## Step 6. The tests

- **Isolation and continuity** run in one process with `InMemorySaver`, with a fresh graph and a fresh `thread_id` per test.
- **The history test** pins the exact steps `[4, 3, 2, 1, 0, -1]` and the message counts for two turns.
- **The restart tests use `subprocess`** to run `chat_cli.py` as a real separate process, with the database in pytest's `tmp_path` so nothing is left behind and no old file can fool the test.
- **A trap test:** `":memory:"` looks like SQLite but forgets, so it documents the bug.

In [8]:
%%writefile test_memory_threads.py
import subprocess
import sys
import uuid
from pathlib import Path

import pytest

from echo_memory import build_graph, make_saver

HERE = Path(__file__).parent


# ---------------------------------------------------------------- fixtures
@pytest.fixture
def graph():
    return build_graph(make_saver("memory"))        # fresh memory per test


def thread(name=None):
    return {"configurable": {"thread_id": name or str(uuid.uuid4())}}


def say(graph, config, text):
    return graph.invoke({"messages": [text]}, config)


def cli(*args):
    """Run chat_cli.py as a separate process: this is a real 'restart'."""
    out = subprocess.run(
        [sys.executable, "chat_cli.py", *args],
        cwd=HERE, capture_output=True, text=True, check=True,
    )
    return out.stdout


# ------------------------------------------- 1. one thread keeps its memory
def test_first_turn_echoes_the_message(graph):
    result = say(graph, thread(), "hello")
    assert result["messages"] == ["hello", "echo (turn 1): hello"]
    assert result["turn_count"] == 1


def test_second_turn_on_same_thread_continues(graph):
    cfg = thread()
    say(graph, cfg, "one")
    result = say(graph, cfg, "two")
    assert result["messages"] == ["one", "echo (turn 1): one", "two", "echo (turn 2): two"]
    assert result["turn_count"] == 2


# ----------------------------------------------- 2. threads are isolated
def test_threads_do_not_share_messages_or_counts(graph):
    a, b = thread("a"), thread("b")
    say(graph, a, "a1")
    say(graph, a, "a2")
    result_b = say(graph, b, "b1")
    assert result_b["messages"] == ["b1", "echo (turn 1): b1"]
    assert result_b["turn_count"] == 1
    assert graph.get_state(a).values["turn_count"] == 2


def test_a_new_thread_id_starts_fresh(graph):
    say(graph, thread("old"), "hi")
    assert say(graph, thread("new"), "hi")["turn_count"] == 1


# ------------------------------------------------------- 3. the history
def test_history_has_one_checkpoint_per_step(graph):
    cfg = thread()
    say(graph, cfg, "one")
    say(graph, cfg, "two")
    history = list(graph.get_state_history(cfg))
    assert [s.metadata["step"] for s in history] == [4, 3, 2, 1, 0, -1]    # newest first
    assert [len(s.values.get("messages", [])) for s in history] == [4, 3, 2, 2, 1, 0]
    assert graph.get_state(cfg).next == ()                                  # run finished


# -------------------------------- 4. restart: the memory saver forgets...
def test_memory_saver_forgets_after_a_restart():
    first = cli("--saver", "memory", "--thread", "t", "hello")
    second = cli("--saver", "memory", "--thread", "t", "again")
    assert "turn_count: 1" in first
    assert "turn_count: 1" in second            # a new process: the first chat is gone


# ------------------------------------ ...and the sqlite saver remembers
def test_sqlite_saver_survives_a_restart(tmp_path):
    db = str(tmp_path / "chat.db")
    first = cli("--saver", "sqlite", "--db", db, "--thread", "t", "hello")
    second = cli("--saver", "sqlite", "--db", db, "--thread", "t", "again")   # a new process
    assert "turn_count: 1" in first
    assert "turn_count: 2" in second
    assert "echo (turn 1): hello" in second     # the first process's message is still there


def test_sqlite_state_can_be_read_by_a_third_process(tmp_path):
    db = str(tmp_path / "chat.db")
    cli("--saver", "sqlite", "--db", db, "--thread", "t", "hello")
    shown = cli("--saver", "sqlite", "--db", db, "--thread", "t", "--show")
    assert "saved turn_count: 1" in shown


def test_sqlite_threads_stay_isolated_across_restarts(tmp_path):
    db = str(tmp_path / "chat.db")
    cli("--saver", "sqlite", "--db", db, "--thread", "a", "hello")
    other = cli("--saver", "sqlite", "--db", db, "--thread", "b", "hello")
    assert "turn_count: 1" in other


def test_in_memory_sqlite_path_is_a_trap():
    """':memory:' looks like sqlite but lives only inside one process."""
    cli("--saver", "sqlite", "--db", ":memory:", "--thread", "t", "hello")
    second = cli("--saver", "sqlite", "--db", ":memory:", "--thread", "t", "again")
    assert "turn_count: 1" in second


def test_unknown_saver_kind_is_rejected():
    with pytest.raises(ValueError, match="unknown saver"):
        make_saver("postgres")

Overwriting test_memory_threads.py


The subprocess tests start Python several times, so this takes a few seconds.

In [9]:
!python -m pytest -v --no-header -p no:cacheprovider

============================= test session starts ==============================
collected 25 items / 2 errors                                                  

==================================== ERRORS ====================================
______ ERROR collecting langgraph_practice_projects/test_bounded_loop.py _______
test_bounded_loop.py:36: in <module>
    @pytest.mark.parameterize(
     ^^^^^^^^^^^^^^^^^^^^^^^^
E   Failed: Unknown 'parameterize' mark, did you mean 'parametrize'?
_ ERROR collecting langgraph_practice_projects/test_bounded_loop_corrected.py __
ImportError while importing test module '/home/gauravgupta/codebase/agentic-ai-krish-naik/langgraph_practice_projects/test_bounded_loop_corrected.py'.
Hint: make sure your test modules/packages have valid Python names.
Traceback:
../../../.local/share/uv/python/cpython-3.12.13-linux-x86_64-gnu/lib/python3.12/importlib/__init__.py:90: in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           

## Step 7. Check that the tests can fail

Two realistic mistakes, run through the same CLI: compiling without a checkpointer, and a SQLite path that is different each run.

In [10]:
from echo_memory import build_graph
no_saver = build_graph()                               # forgot checkpointer=...
cfg = {"configurable": {"thread_id": "t"}}
for text in ["hi", "again", "and again"]:
    r = no_saver.invoke({"messages": [text]}, cfg)
print("no checkpointer, turn_count after 3 calls:", r["turn_count"], "(a working saver would say 3)")
print("messages:", r["messages"])

no checkpointer, turn_count after 3 calls: 1 (a working saver would say 3)
messages: ['and again', 'echo (turn 1): and again']


In [11]:
print("--- different database file each run (a path bug) ---")
!{PY} chat_cli.py --saver sqlite --db {workdir}/run_one.db --thread t "hello"
!{PY} chat_cli.py --saver sqlite --db {workdir}/run_two.db --thread t "again"

--- different database file each run (a path bug) ---
messages: ['hello', 'echo (turn 1): hello']
turn_count: 1
messages: ['again', 'echo (turn 1): again']
turn_count: 1


## Your turn

1. Restart the kernel and run all cells. Everything should pass.
2. Run the two-terminal check (Step 5A) yourself, once with each saver. Say out loud what you expect before each run.
3. Add a `--history` option to `chat_cli.py` that prints the step table from Step 3 for a thread. Write a test that runs it after two turns.
4. Add to `stumbles.txt` anything you hesitated on: "what is a restart, exactly?", "what does `next` mean in a snapshot?", "why did the second run show 4 messages?"